# R18-H196 + R18-H197 - dropping NLI, and snapshot pinning

**H196** widens the H194 prose/feature stratum to >= 30 balanced pairs (blind adjudication FIRST - labels were assigned by
reading each rendered context against the gold BEFORE any scorer ran; frozen in `data/processed/instrument-prose-bench-h196.json`).
Adds a stop-word-filtered word-overlap scorer, re-runs the matrix (comparator / filtered-overlap / raw-overlap / NLI).
Bar: filtered overlap >= NLI on the prose stratum AND combined-instrument router_gpufree >= 0.97; refuted if NLI holds a
>= 5pt edge on any n>=10 stratum. NLI = mDeBERTa fp16 on GPU 2 (eager attention), CPU fallback.

**H197** implements a graph snapshot fingerprint (node/edge/embedding counts + content hash over gold-carrier renders +
embedding digest, < 5s) and runs the re-adjudication replay: 60/60 reproduction on a matched fingerprint, guaranteed
mismatch detection on a mutated state (a doc's carriers excluded from the render). `neo4j2` READ-ONLY, no writes.

## H196 setup - NLI (GPU 2), graph render, and the four scorers

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"; os.environ["CUDA_VISIBLE_DEVICES"]="2"; os.environ["HF_HUB_OFFLINE"]="1"
os.environ["NEO4J_URI"]="bolt://user-konrad.jelen-kgf-neo4j2:7687"
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
NLI_MODEL="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok=AutoTokenizer.from_pretrained(NLI_MODEL); DEV="cuda"
try:
    nli_model=AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,dtype=torch.float16,attn_implementation="eager").to("cuda").eval()
except RuntimeError:
    DEV="cpu"; nli_model=AutoModelForSequenceClassification.from_pretrained(NLI_MODEL,dtype=torch.float32,attn_implementation="eager").to("cpu").eval()
ENT_IDX=[i for i,l in nli_model.config.id2label.items() if l=="entailment"][0]
print(f"NLI on {DEV} dtype={next(nli_model.parameters()).dtype}")
import re, json, unicodedata, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np
from neo4j import GraphDatabase
d=GraphDatabase.driver(os.environ["NEO4J_URI"], auth=("neo4j","kgfoundry"))
with d.session() as s:
    ents=s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,properties(e) AS props,labels(e) AS types").data()
    edges=s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    prop_rows=s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid,p.text AS text").data()
    alias_rows=s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id RETURN e.id AS eid,collect(DISTINCT a.id)[..5] AS aliases").data()
d.close()
node={r["id"]:r for r in ents}; names={r["id"]:r["name"] for r in ents}
props_by=defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by={r["eid"]:r["aliases"] for r in alias_rows}
rels_by=defaultdict(list)
for e in edges: rels_by[e["a"]].append((e["rel"],e["b"])); rels_by[e["b"]].append((e["rel"],e["a"]))
def spec_of(r): return {k.removeprefix("prop_"):v for k,v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r=node[nid]; spec=dict(spec_of(r))
    for a in [a for a in alias_by.get(nid,[]) if a in node]:
        for k,v in spec_of(node[a]).items(): spec.setdefault(k,v)
    return spec
def base_render(nid):
    r=node[nid]; spec=merged_spec(nid); al=[a for a in alias_by.get(nid,[]) if a in node]
    aka=(f"Also known as: {', '.join(names.get(a,'') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid): return base_render(nid)+" "+" ; ".join(f"{t} -> {names.get(b,'')}" for t,b in rels_by.get(nid,[])[:15])
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n,[])]
_TM=dict.fromkeys(map(ord,"®™©"),None)
def gnorm(s):
    s=(s or "").translate(_TM); s=unicodedata.normalize("NFKC",s)
    s=s.replace(" "," ").replace("×","x").replace("*","x").replace("·","x"); s=re.sub(r"(?<=\d),(?=\d)","",s)
    return re.sub(r"\s+"," ",s.casefold()).strip()
STOP={"the","a","an","of","for","and","or","to","in","on","with","is","are","be","that","this","it","its",
      "does","do","offer","offers","what","which","support","supported","supports","feature","features",
      "have","has","your","you","during","from","by","as","at","per","up","down","her","his","their","not"}
def _words(t): return set(re.findall(r"[a-z][a-z0-9\-]{2,}", gnorm(t)))
def word_overlap(gold, ids, thr=0.6, filt=False):
    w=_words(gold); cw=_words(" ".join(units_of(ids)))
    if filt: w=w-STOP
    return bool(w) and len(w&cw)/len(w)>=thr
# comparator (verbatim H194) - abstains (None) on prose/feature
UNITWORD={"mm":"len_mm","cm":"len_cm","g":"mass_g","kg":"mass_kg","oz":"mass_oz","ml":"vol_ml","l":"vol_l","db":"sound_db","dba":"sound_db","w":"power_w","hz":"freq_hz","cmh2o":"press","m":"alt_m","min":"time_min","mins":"time_min","minute":"time_min","minutes":"time_min","year":"warr_y","years":"warr_y"}
FAM_EQ={"len_mm":{"len_mm"},"vol_ml":{"vol_ml","vol_l"},"mass_kg":{"mass_kg"},"mass_g":{"mass_g"},"sound_db":{"sound_db"},"power_w":{"power_w"},"time_min":{"time_min"},"warr_y":{"warr_y"},"press":{"press"},"freq_hz":{"freq_hz"},"alt_m":{"alt_m"},"len_cm":{"len_cm"},"mass_oz":{"mass_oz"}}
def key_family(k):
    k=k.lower()
    if "dimension" in k or re.search(r"_mm\b",k): return "len_mm"
    if k.endswith("_kg") or "weight_kg" in k: return "mass_kg"
    if "sound" in k or re.search(r"_db\b",k) or "noise" in k: return "sound_db"
    if "power" in k: return "power_w"
    if "ramp" in k or "delay" in k: return "time_min"
    if "warranty" in k: return "warr_y"
    if "pressure" in k: return "press"
    return None
def nums_in(v): return re.findall(r"\d+(?:\.\d+)?", str(v).replace(",",""))
def ctx_quantities(ids):
    Q=set()
    for nid in ids:
        spec=merged_spec(nid)
        for k,v in spec.items():
            fam=key_family(k)
            if fam:
                for n in nums_in(v): Q.add((n,fam))
        text=gnorm(seed_render(nid)+" "+" ".join(props_by.get(nid,[])))
        for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", text):
            fam=UNITWORD.get(m.group(2))
            if fam: Q.add((m.group(1),fam))
    return Q
def parse_gold(gold):
    g=gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return ("dim", re.findall(r"\d+(?:\.\d+)?", g))
    m=re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", g)
    if m: return ("num",(m.group(1),UNITWORD.get(m.group(2))))
    return ("other", None)
def comparator(gold, ids):
    kind,payload=parse_gold(gold)
    if kind=="other": return None
    Q=ctx_quantities(ids)
    if kind=="num":
        n,fam=payload
        if fam is None: return any(x==n for x,_ in Q)
        eq=FAM_EQ.get(fam,{fam}); return any(x==n and f in eq for x,f in Q)
    if kind=="dim":
        a,b,c=payload; mm={n for n,f in Q if f=="len_mm"}; return {a,b,c}<=mm
    return None
@torch.inference_mode()
def nli_entail(gold, units, bs=48):
    hl=len(nli_tok(gold, add_special_tokens=False)["input_ids"]); L=512-hl-3; step=max(32,L-40)
    prem=[]
    for u in units:
        pids=nli_tok(u, add_special_tokens=False)["input_ids"]
        wins=[pids] if len(pids)<=L else [pids[i:i+L] for i in range(0,len(pids),step)]
        for w in wins: prem.append(nli_tok.decode(w))
    if not prem: return 0.0
    best=0.0
    for i in range(0,len(prem),bs):
        b=prem[i:i+bs]
        enc=nli_tok(b,[gold]*len(b),return_tensors="pt",truncation="only_first",padding=True,max_length=512).to(DEV)
        best=max(best, torch.softmax(nli_model(**enc).logits.float(),-1)[:,ENT_IDX].max().item())
    return best

print("harness + scorers ready")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:  49%|████▉     | 99/202 [00:00<00:00, 984.55it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1481.21it/s]

NLI on cuda dtype=torch.float16


harness + scorers ready


### The blind-adjudicated bench and the scorer matrix

In [2]:
import datetime
STAMP=datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
B=json.load(open("../data/processed/instrument-prose-bench-h196.json"))
pairs=B["pairs"]                          # frozen: blind labels adjudicated by reading renders BEFORE scoring
for p in pairs: p["label"]=int(p["label"])
print(f"loaded frozen prose/feature bench: n={len(pairs)} present={sum(p['label'] for p in pairs)} absent={sum(1-p['label'] for p in pairs)}")
nli_sc={p["i"]:nli_entail(p["gold"], units_of(p["ctx_ids"])) for p in pairs}
preds={
 "comparator":{p["i"]:(int(bool(comparator(p["gold"],p["ctx_ids"]))) if comparator(p["gold"],p["ctx_ids"]) is not None else 0) for p in pairs},
 "raw_overlap":{p["i"]:int(word_overlap(p["gold"],p["ctx_ids"],filt=False)) for p in pairs},
 "filtered_overlap":{p["i"]:int(word_overlap(p["gold"],p["ctx_ids"],filt=True)) for p in pairs},
 "nli@0.5":{p["i"]:int(nli_sc[p["i"]]>=0.5) for p in pairs},
}
def routed(p):
    c=comparator(p["gold"],p["ctx_ids"]); return int(bool(c)) if c is not None else int(word_overlap(p["gold"],p["ctx_ids"],filt=True))
preds["router_gpufree"]={p["i"]:routed(p) for p in pairs}
def agree(pred, subset=None):
    ps=[p for p in pairs if subset is None or p["stratum"]==subset]
    return float(np.mean([pred[p["i"]]==p["label"] for p in ps])) if ps else None
strata=["prose","feature"]
print(f"\n=== H196 matrix on widened prose/feature bench (n={len(pairs)}: prose {sum(1 for p in pairs if p['stratum']=='prose')}, feature {sum(1 for p in pairs if p['stratum']=='feature')}) ===")
print(f"{'scorer':18s} {'overall':>8s} " + " ".join(f"{s:>9s}" for s in strata))
for name,pred in preds.items():
    print(f"{name:18s} {agree(pred):>8.3f} " + " ".join(f"{(agree(pred,s) or 0):>9.3f}" for s in strata))
print("\nNLI - filtered_overlap edge per stratum (refute H196 if NLI >= +5pt on any n>=10 stratum):")
edges={}
for s in strata+["ALL"]:
    sub=None if s=="ALL" else s; n=len([p for p in pairs if sub is None or p["stratum"]==sub])
    e=(agree(preds["nli@0.5"],sub) or 0)-(agree(preds["filtered_overlap"],sub) or 0); edges[s]=e*100
    print(f"  {s:8s} n={n:2d}  NLI-filtered = {e*100:+.1f}pt")
# combined instrument router_gpufree: H194 numeric+dim (comparator=1.0, 49 pairs) + widened prose/feature
h194=json.load(open(f"../reports/instrument-bench-h194-20260707T144609Z.json"))["agreement_matrix"]
n_numdim=49; comp_numdim=1.0; n_pf=len(pairs); rgf_pf=agree(preds["router_gpufree"])
combined_rgf=(n_numdim*comp_numdim + n_pf*rgf_pf)/(n_numdim+n_pf)
print(f"\ncombined-instrument router_gpufree = (49*1.00 + {n_pf}*{rgf_pf:.3f})/{n_numdim+n_pf} = {combined_rgf:.3f}  (H194 orig router_gpufree overall={h194['router_gpufree']['overall']:.3f})")
filt_ge_nli_prose = agree(preds["filtered_overlap"],"prose") >= agree(preds["nli@0.5"],"prose")
nli_holds_edge = any(edges[s]>=5.0 for s in strata if len([p for p in pairs if p["stratum"]==s])>=10)
print(f"\nBARS: filtered_overlap >= NLI on prose = {filt_ge_nli_prose} | NLI holds >=5pt edge (refute) = {nli_holds_edge} | router_gpufree>=0.97 = {combined_rgf>=0.97}")
rep=dict(round="R18-H196",utc=STAMP,graph="neo4j2",n=len(pairs),
    agreement={k:{"overall":agree(v)}|{s:agree(v,s) for s in strata} for k,v in preds.items()},
    nli_minus_filtered_pts=edges, combined_instrument_router_gpufree=combined_rgf,
    bars=dict(filtered_ge_nli_prose=filt_ge_nli_prose, nli_holds_5pt_edge=nli_holds_edge, router_gpufree_ge_097=combined_rgf>=0.97))
Path(f"../reports/instrument-router-h196-{STAMP}.json").write_text(json.dumps(rep,indent=2))
print(f"saved reports/instrument-router-h196-{STAMP}.json")


loaded frozen prose/feature bench: n=48 present=21 absent=27


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (893 > 512). Running this sequence through the model will result in indexing errors



=== H196 matrix on widened prose/feature bench (n=48: prose 12, feature 36) ===
scorer              overall     prose   feature
comparator            0.562     0.667     0.528
raw_overlap           0.792     0.833     0.778
filtered_overlap      0.812     0.917     0.778
nli@0.5               0.604     0.667     0.583
router_gpufree        0.812     0.917     0.778

NLI - filtered_overlap edge per stratum (refute H196 if NLI >= +5pt on any n>=10 stratum):
  prose    n=12  NLI-filtered = -25.0pt
  feature  n=36  NLI-filtered = -19.4pt
  ALL      n=48  NLI-filtered = -20.8pt

combined-instrument router_gpufree = (49*1.00 + 48*0.812)/97 = 0.907  (H194 orig router_gpufree overall=0.967)

BARS: filtered_overlap >= NLI on prose = True | NLI holds >=5pt edge (refute) = False | router_gpufree>=0.97 = False
saved reports/instrument-router-h196-20260707T152431Z.json


## H197 - graph snapshot fingerprint and re-adjudication replay

In [3]:
import os, warnings, re, json, hashlib, time
warnings.filterwarnings("ignore")
os.environ["NEO4J_URI"]="bolt://user-konrad.jelen-kgf-neo4j2:7687"
from pathlib import Path
from collections import defaultdict
from neo4j import GraphDatabase
d=GraphDatabase.driver(os.environ["NEO4J_URI"], auth=("neo4j","kgfoundry"))
with d.session() as s:
    ents=s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,properties(e) AS props,labels(e) AS types, e.source_documents AS sd").data()
    edges=s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    embs=s.run("MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()
    docmap={r["id"]:r["nm"] for r in s.run("MATCH (dd:KGFDocument) RETURN dd.id AS id, dd.name AS nm").data()}
d.close()
node={r["id"]:r for r in ents}; names={r["id"]:r["name"] for r in ents}
rels_by=defaultdict(list)
for e in edges: rels_by[e["a"]].append((e["rel"],e["b"])); rels_by[e["b"]].append((e["rel"],e["a"]))
emb_head={r["id"]:r["head"] for r in embs}
def seed_render(nid):
    r=node.get(nid)
    if r is None: return ""
    spec={k.removeprefix("prop_"):v for k,v in r["props"].items() if k.startswith("prop_")}
    return f"## {r['name']} ({', '.join(r['types'])})\n{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)} "+ \
           " ; ".join(f"{t} -> {names.get(b,'')}" for t,b in rels_by.get(nid,[])[:15])

bench=json.load(open("../data/processed/instrument-bench-h194.json"))["pairs"]
carriers=sorted({nid for p in bench for nid in p["ctx_ids"] if nid in node})
print(f"bench pairs {len(bench)} | gold-carrier entities {len(carriers)}")

def fingerprint(carrier_ids, exclude=frozenset()):
    """node/edge/embedding counts + content hash over gold-carrier renders + embedding digest. Target < 5s."""
    cids=[c for c in carrier_ids if c not in exclude]
    render_blob="\x1e".join(seed_render(c) for c in cids)
    emb_blob=";".join(f"{c}:"+",".join(f"{x:.4f}" for x in emb_head.get(c,[])) for c in cids)
    return dict(
        node_count=len(node), edge_count=len(edges), embedding_count=len(emb_head),
        n_carriers=len(cids),
        content_hash=hashlib.sha256(render_blob.encode()).hexdigest()[:16],
        embedding_digest=hashlib.sha256(emb_blob.encode()).hexdigest()[:16])

t0=time.time(); fp0=fingerprint(carriers); dt=time.time()-t0
print(f"fingerprint computed in {dt*1000:.0f} ms (<5s bar: {'PASS' if dt<5 else 'FAIL'})")
print("fp0:", json.dumps(fp0))

# deterministic adjudication A(graph): present/absent per pair via fuzzy over current renders of ctx_ids
def _norm(s): return re.sub(r"\s+"," ",(s or "").casefold())
def value_tokens(t): return re.findall(r"[\w.\-/]*\d[\w.\-/]*", t)
_UNIT=r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"
def fuzzy_present(gold, ctx):
    ng=_norm(gold)
    if ng in ctx: return True
    sq=re.sub(r"[\s,()]","",ctx); sk=re.sub(r"[\s,()]","",re.sub(_UNIT,"",ng))
    if any(c.isdigit() for c in sk) and len(sk)>=5 and sk in sq: return True
    tok=value_tokens(gold)
    if tok:
        hit=sum(1 for t in tok if _norm(t) in ctx or re.sub(r"[\s,()]","",_norm(t)) in sq)
        return hit>=max(1,len(tok)//2+(len(tok)%2))
    w=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ng)); cw=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ctx))
    return bool(w) and len(w&cw)/len(w)>=0.6
def adjudicate(exclude=frozenset()):
    A={}
    for p in bench:
        ids=[i for i in p["ctx_ids"] if i in node and i not in exclude]
        ctx=_norm(" ".join(seed_render(i) for i in ids))
        A[p["i"]]=int(fuzzy_present(p["gold"], ctx))
    return A
A0=adjudicate()

# --- matched replay: recompute fp + adjudication on unchanged graph ---
fp1=fingerprint(carriers); A1=adjudicate()
matched = fp1==fp0
repro = sum(1 for i in A0 if A0[i]==A1[i])
print(f"\nMATCHED replay: fingerprint match={matched} | adjudication reproduction {repro}/{len(A0)}")

# --- mutation: exclude one doc's carrier entities (simulates a doc drop / re-embed shifting the surface) ---
# pick the doc contributing the most carrier entities
doc_of=lambda nid: [docmap.get(x) for x in (node[nid].get("sd") or []) if docmap.get(x)]
from collections import Counter
doc_carriers=Counter()
for c in carriers:
    for dn in set(doc_of(c)): doc_carriers[dn]+=1
target_doc,_=doc_carriers.most_common(1)[0]
excl={c for c in carriers if target_doc in doc_of(c)}
fp_mut=fingerprint(carriers, exclude=excl); A_mut=adjudicate(exclude=excl)
mismatch = fp_mut!=fp0
drifted = sum(1 for i in A0 if A0[i]!=A_mut[i])
print(f"MUTATION (exclude doc {target_doc!r}, {len(excl)} carriers): fingerprint mismatch={mismatch} | "
      f"content_hash {fp0['content_hash']}->{fp_mut['content_hash']} | adjudication drift on {drifted} pairs")
print(f"\nBARS: matched-reproduction 60/60 = {repro==len(A0)} | mutation-mismatch-fires-first = {mismatch}")
h197=dict(round="R18-H197",utc=STAMP if 'STAMP' in dir() else __import__("datetime").datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
    graph="neo4j2", fingerprint_ms=round(dt*1000,1), fingerprint_under_5s=bool(dt<5),
    fp0=fp0, matched_reproduction=f"{repro}/{len(A0)}", matched_ok=bool(repro==len(A0)),
    mutation=dict(excluded_doc=target_doc, excluded_carriers=len(excl), fingerprint_mismatch=bool(mismatch),
                  content_hash_before=fp0["content_hash"], content_hash_after=fp_mut["content_hash"], adjudication_drift_pairs=drifted),
    bars=dict(reproduces_60_of_60=bool(repro==len(A0)), mutation_mismatch_fires=bool(mismatch)))
from pathlib import Path as _P
_st=h197["utc"]; _P(f"../reports/bench-fingerprint-h197-{_st}.json").write_text(json.dumps(h197,indent=2))
print(f"saved reports/bench-fingerprint-h197-{_st}.json")


bench pairs 60 | gold-carrier entities 153
fingerprint computed in 2 ms (<5s bar: PASS)
fp0: {"node_count": 2798, "edge_count": 3905, "embedding_count": 2798, "n_carriers": 153, "content_hash": "5d5632aeaa92f48d", "embedding_digest": "c03a17c57c5fe1ee"}

MATCHED replay: fingerprint match=True | adjudication reproduction 60/60
MUTATION (exclude doc 'ResMed-Airsense-11-Manual.pdf', 40 carriers): fingerprint mismatch=True | content_hash 5d5632aeaa92f48d->111fd0bf905c0a62 | adjudication drift on 6 pairs

BARS: matched-reproduction 60/60 = True | mutation-mismatch-fires-first = True
saved reports/bench-fingerprint-h197-20260707T152431Z.json
